# Rebound State Factor Mining v1

Baselines:
- Canonical comparison: `stable_rebound_label_discovery_v1`.
- Source frontier card: `state_head_stable_rebound_label_screen_v1`.
- Source target table: `state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv`.

Role: `diagnose` and target-contract handoff. This notebook does not train or promote an actor.

Changed layer: `feature` / `selection`.

Hypothesis: a rebound state should require a short-term SOXL pullback plus confirmed SOXL/SOXX relative-strength context. It should improve forward returns across duplicate fold-native target rows, recur across years in full history, and avoid the standalone drawdown failure seen in the older deep-capitulation label.

Acceptance criteria:
- Target-table label stability: duplicate-date conflict rate is zero, at least 6 positive rows, at least 2 fold windows, at least 2 positives in each covered fold.
- Target-table return lift: 5-day lift at least 3 percentage points, with non-negative 10-day lift.
- Recurrence: at least 8 unique target dates, at least 2 target years, at least 250 target span days, and full-history signals across at least 5 years and 30 rows.
- Risk sanity: 5-day standalone hold MaxDD must be no worse than -20% for a direct handoff. If this only works under a cap, route as risk-capped overlay, not direct core feature.

Failure mode: a candidate that is only a single-crisis bottom label, has duplicate-date conflicts, or only works with very high standalone drawdown is diagnostic only.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate_root in (ROOT, *ROOT.parents):
    if (candidate_root / "src" / "soxl_quant").exists():
        ROOT = candidate_root
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.stable_label_discovery import (
    BinaryLabelDiscoveryConfig,
    score_binary_label_candidates,
)

EXPERIMENT = "rebound_state_factor_mining_v1"
CARD_DIR = ROOT / "runtime" / "factor" / "card" / EXPERIMENT
PANEL_PATH = ROOT / "data" / "processed" / "panels" / "soxl_soxx_vix_full.csv"
TARGET_PATH = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_native_teacher_router_v1" / "fold_native_teacher_targets.csv"
PREVIOUS_FACTOR_CARD = ROOT / "runtime" / "factor" / "card" / "stable_rebound_label_discovery_v1" / "summary.json"
SOURCE_FRONTIER = "state_head_stable_rebound_label_screen_v1"

CARD_DIR.mkdir(parents=True, exist_ok=True)
print(f"card_dir={CARD_DIR}")
print(f"panel_exists={PANEL_PATH.exists()} target_exists={TARGET_PATH.exists()} previous_factor_exists={PREVIOUS_FACTOR_CARD.exists()}")

card_dir=/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/factor/card/rebound_state_factor_mining_v1
panel_exists=True target_exists=True previous_factor_exists=True


In [2]:
def max_drawdown(nav: pd.Series) -> float:
    peak = nav.cummax()
    drawdown = nav / peak - 1.0
    return float(drawdown.min()) if len(drawdown) else float("nan")


def annualized_sharpe(returns: pd.Series) -> float:
    returns = returns.dropna()
    if returns.std() == 0 or len(returns) < 2:
        return float("nan")
    return float(returns.mean() / returns.std() * np.sqrt(252))


def build_full_history_features(panel_path: Path) -> pd.DataFrame:
    panel = pd.read_csv(panel_path, parse_dates=["date"]).sort_values("date").set_index("date")
    close = panel["soxl_close"].astype(float)
    soxx = panel["soxx_close"].astype(float)
    vix = panel["vix"].astype(float)
    volume = panel["soxl_volume"].astype(float)
    ret = close.pct_change()
    soxx_ret = soxx.pct_change()
    ratio = close / soxx.replace(0.0, np.nan)

    features = pd.DataFrame(index=panel.index)
    features["close"] = close
    features["return"] = ret
    for window in (5, 10, 21, 63, 126):
        features[f"mom_{window}"] = ret.rolling(window).sum()
    for window in (21, 63, 126):
        features[f"drawdown_{window}"] = close / close.rolling(window).max() - 1.0
    features["dd63_repair_5"] = features["drawdown_63"] - features["drawdown_63"].shift(5)
    features["dd63_repair_10"] = features["drawdown_63"] - features["drawdown_63"].shift(10)
    features["rv_21"] = ret.rolling(21).std() * np.sqrt(252)
    features["rv_63"] = ret.rolling(63).std() * np.sqrt(252)
    features["rv21_change_5"] = features["rv_21"] - features["rv_21"].shift(5)
    features["vol_of_vol_21"] = features["rv_21"].rolling(21).std()
    features["ratio_mom_5"] = ratio.pct_change(5)
    features["ratio_mom_10"] = ratio.pct_change(10)
    features["ratio_mom_21"] = ratio.pct_change(21)
    features["ratio_z_252"] = (ratio - ratio.rolling(252).mean()) / (ratio.rolling(252).std() + 1e-8)
    features["ratio_widen_21"] = features["ratio_mom_21"]
    features["soxx_mom_5"] = soxx_ret.rolling(5).sum()
    features["soxx_mom_10"] = soxx_ret.rolling(10).sum()
    features["soxx_mom_21"] = soxx_ret.rolling(21).sum()
    features["vix_z_252"] = (vix - vix.rolling(252).mean()) / (vix.rolling(252).std() + 1e-8)
    features["vix_drop_5"] = -vix.pct_change(5)
    features["volume_z_63"] = (volume - volume.rolling(63).mean()) / (volume.rolling(63).std() + 1e-8)
    for horizon in (5, 10, 21):
        features[f"forward_return_{horizon}"] = close.shift(-horizon) / close - 1.0
    return features.replace([np.inf, -np.inf], np.nan)


full_features = build_full_history_features(PANEL_PATH)
full_features.tail()

,close,return,mom_5,mom_10,mom_21,mom_63,mom_126,drawdown_21,drawdown_63,drawdown_126,...,ratio_widen_21,soxx_mom_5,soxx_mom_10,soxx_mom_21,vix_z_252,vix_drop_5,volume_z_63,forward_return_5,forward_return_10,forward_return_21
date,,,,,,,,,,,,,,,,,,,,,
2026-06-04,262.70,-0.063592,0.173563,0.452544,0.691408,1.789326,2.137026,-0.063592,-0.063592,-0.063592,...,0.459500,0.058749,0.151242,0.232716,-0.818774,0.021601,-0.655829,NaN,NaN,NaN
2026-06-05,182.54,-0.305139,-0.130285,0.117440,0.235811,1.610282,1.768417,-0.349326,-0.349326,-0.349326,...,0.033547,-0.045009,0.038342,0.078266,1.030286,-0.404047,1.268726,NaN,NaN,NaN
2026-06-08,211.44,0.158321,0.016046,0.207540,0.477039,1.655219,1.957224,-0.246311,-0.246311,-0.246311,...,0.197740,0.008675,0.072983,0.165584,0.244660,-0.178816,-0.275063,NaN,NaN,NaN
2026-06-09,201.68,-0.046160,-0.203174,-0.023496,0.267565,1.585241,1.881392,-0.281101,-0.281101,-0.281101,...,0.054985,-0.065474,-0.004277,0.092545,0.528745,-0.259987,1.927406,NaN,NaN,NaN
2026-06-10,180.65,-0.104274,-0.360843,-0.093181,0.087107,1.453489,1.745075,-0.356063,-0.356063,-0.356063,...,-0.066637,-0.119792,-0.030259,0.031899,1.231218,-0.383562,0.752044,NaN,NaN,NaN


In [3]:
def qvalue(quantiles: pd.DataFrame, column: str, quantile: float) -> float:
    return float(quantiles.loc[quantile, column])


def build_candidate_masks(features: pd.DataFrame) -> dict[str, pd.Series]:
    quantile_columns = [
        "mom_5",
        "mom_10",
        "mom_21",
        "drawdown_63",
        "drawdown_126",
        "rv_21",
        "ratio_mom_10",
        "ratio_mom_21",
        "soxx_mom_10",
        "soxx_mom_21",
        "vix_z_252",
        "dd63_repair_5",
        "rv21_change_5",
    ]
    qs = features[quantile_columns].quantile([0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80])
    masks: dict[str, pd.Series] = {}

    for ddp in (0.10, 0.15, 0.20, 0.25, 0.30):
        for momp in (0.20, 0.30, 0.40, 0.50, 0.60):
            masks[f"deep_capitulation__mom5_leq{momp:.2f}_dd63_leq{ddp:.2f}"] = (
                (features["mom_5"] <= qvalue(qs, "mom_5", momp))
                & (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
            )

    for ddp in (0.10, 0.15, 0.20, 0.25, 0.30):
        for repairp in (0.60, 0.70, 0.75, 0.80):
            masks[f"dd_repair__dd63_leq{ddp:.2f}_repair5_geq{repairp:.2f}"] = (
                (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
                & (features["dd63_repair_5"] >= qvalue(qs, "dd63_repair_5", repairp))
            )
        for momp in (0.50, 0.60, 0.70, 0.75):
            masks[f"mom_reentry__dd63_leq{ddp:.2f}_mom5_geq{momp:.2f}"] = (
                (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
                & (features["mom_5"] >= qvalue(qs, "mom_5", momp))
            )
        for soxxp in (0.50, 0.60, 0.70, 0.75):
            masks[f"soxx_confirmed_reentry__dd63_leq{ddp:.2f}_soxx10_geq{soxxp:.2f}"] = (
                (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
                & (features["soxx_mom_10"] >= qvalue(qs, "soxx_mom_10", soxxp))
            )
        for ratiop in (0.60, 0.70, 0.75):
            masks[f"ratio_recovery__dd63_leq{ddp:.2f}_ratio10_geq{ratiop:.2f}"] = (
                (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
                & (features["ratio_mom_10"] >= qvalue(qs, "ratio_mom_10", ratiop))
            )

    for ddp in (0.10, 0.15, 0.20, 0.25, 0.30, 0.40):
        for rvp in (0.10, 0.20, 0.30, 0.40):
            masks[f"vol_release__dd63_leq{ddp:.2f}_rvchg5_leq{rvp:.2f}"] = (
                (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
                & (features["rv21_change_5"] <= qvalue(qs, "rv21_change_5", rvp))
                & (features["mom_5"] >= qvalue(qs, "mom_5", 0.40))
            )
        for vixp in (0.60, 0.70, 0.80):
            masks[f"vix_release__dd63_leq{ddp:.2f}_vixdrop5_geq{vixp:.2f}"] = (
                (features["drawdown_63"] <= qvalue(qs, "drawdown_63", ddp))
                & (features["vix_drop_5"] >= features["vix_drop_5"].quantile(vixp))
            )

    for mom5p in (0.10, 0.15, 0.20, 0.25, 0.30):
        for mom21p in (0.60, 0.70, 0.75, 0.80):
            masks[f"trend_pullback_rebound__mom5_leq{mom5p:.2f}_mom21_geq{mom21p:.2f}"] = (
                (features["mom_5"] <= qvalue(qs, "mom_5", mom5p))
                & (features["mom_21"] >= qvalue(qs, "mom_21", mom21p))
            )
        for ratiop in (0.60, 0.70, 0.75, 0.80):
            masks[f"spread_confirmed_pullback__mom5_leq{mom5p:.2f}_ratio21_geq{ratiop:.2f}"] = (
                (features["mom_5"] <= qvalue(qs, "mom_5", mom5p))
                & (features["ratio_mom_21"] >= qvalue(qs, "ratio_mom_21", ratiop))
            )
    return {name: mask.fillna(False).astype(bool) for name, mask in masks.items()}


candidate_masks_full = build_candidate_masks(full_features)
print(f"candidate_count={len(candidate_masks_full)}")

candidate_count=182


In [4]:
def build_target_frame(target_path: Path, features: pd.DataFrame) -> pd.DataFrame:
    target = pd.read_csv(target_path, parse_dates=["datetime"])
    target["date"] = target["datetime"].dt.normalize()
    for horizon in (5, 10, 21):
        target[f"forward_return_{horizon}"] = target["date"].map(features[f"forward_return_{horizon}"])
    target["target_position_lift"] = (
        pd.to_numeric(target["rebound_target_position"], errors="coerce")
        - pd.to_numeric(target["base_position"], errors="coerce")
    ).clip(lower=0.0)
    return target


def target_masks_from_full(target: pd.DataFrame, masks: dict[str, pd.Series]) -> dict[str, pd.Series]:
    mapped: dict[str, pd.Series] = {}
    for name, mask in masks.items():
        mapped[name] = target["date"].map(mask.reindex(full_features.index).fillna(False)).fillna(False).astype(bool)
    return mapped


def add_recurrence_metrics(scores: pd.DataFrame, target: pd.DataFrame, masks: dict[str, pd.Series]) -> pd.DataFrame:
    rows = []
    for name, mask in masks.items():
        dates = target.loc[mask, "date"].drop_duplicates().sort_values()
        rows.append(
            {
                "candidate": name,
                "unique_positive_dates": int(len(dates)),
                "positive_years": int(dates.dt.year.nunique()) if len(dates) else 0,
                "positive_span_days": int((dates.max() - dates.min()).days) if len(dates) > 1 else 0,
            }
        )
    out = scores.merge(pd.DataFrame(rows), on="candidate", how="left")
    out["recurrence_gate"] = (
        (out["unique_positive_dates"] >= 8)
        & (out["positive_years"] >= 2)
        & (out["positive_span_days"] >= 250)
    )
    out["nonnegative_10d_gate"] = out["forward_return_10_lift"] >= 0.0
    out["discovery_gate"] = out["label_gate"] & out["recurrence_gate"] & out["nonnegative_10d_gate"]
    return out


target_frame = build_target_frame(TARGET_PATH, full_features)
candidate_masks_target = target_masks_from_full(target_frame, candidate_masks_full)
config = BinaryLabelDiscoveryConfig(
    min_positive_rows=6,
    min_positive_rate=0.002,
    max_positive_rate=0.35,
    min_fold_coverage=2,
    min_fold_positive_rows=2,
    min_forward_return_lift=0.03,
    min_target_lift=0.0,
    max_label_conflict_rate=0.0,
)
label_result = score_binary_label_candidates(
    target_frame,
    candidate_masks_target,
    group_col="date",
    fold_col="fold",
    forward_return_cols=["forward_return_5", "forward_return_10", "forward_return_21"],
    target_lift_col="target_position_lift",
    config=config,
)
target_label_scores = add_recurrence_metrics(
    label_result.candidate_scores,
    target_frame,
    candidate_masks_target,
)
target_fold_metrics = label_result.fold_metrics.copy()

target_label_scores.head(12)

,candidate,rows,groups,positive_rows,positive_groups,positive_rate,label_conflict_groups,label_conflict_rate,forward_return_5_positive_mean,forward_return_5_negative_mean,...,target_gate,fold_gate,label_gate,label_loss,unique_positive_dates,positive_years,positive_span_days,recurrence_gate,nonnegative_10d_gate,discovery_gate
0,spread_confirmed_pullback__mom5_leq0.15_ratio2...,1261,756,7,4,0.005551,0,0.0,0.186359,0.012632,...,True,True,True,0.0,4,2,840,False,True,False
1,spread_confirmed_pullback__mom5_leq0.15_ratio2...,1261,756,9,5,0.007137,0,0.0,0.182955,0.012379,...,True,True,True,0.0,5,2,840,False,True,False
2,trend_pullback_rebound__mom5_leq0.15_mom21_geq...,1261,756,12,7,0.009516,0,0.0,0.140905,0.012374,...,True,True,True,0.0,7,2,960,False,True,False
3,spread_confirmed_pullback__mom5_leq0.10_ratio2...,1261,756,7,4,0.005551,0,0.0,0.132936,0.012931,...,True,True,True,0.0,4,3,960,False,True,False
4,spread_confirmed_pullback__mom5_leq0.20_ratio2...,1261,756,20,11,0.015860,0,0.0,0.129258,0.011733,...,True,True,True,0.0,11,3,842,True,True,True
5,spread_confirmed_pullback__mom5_leq0.15_ratio2...,1261,756,14,8,0.011102,0,0.0,0.125330,0.012342,...,True,True,True,0.0,8,3,960,True,True,True
6,spread_confirmed_pullback__mom5_leq0.20_ratio2...,1261,756,18,10,0.014274,0,0.0,0.124615,0.011989,...,True,True,True,0.0,10,3,842,True,True,True
7,trend_pullback_rebound__mom5_leq0.20_mom21_geq...,1261,756,21,12,0.016653,0,0.0,0.118529,0.011820,...,True,True,True,0.0,12,3,960,True,True,True
8,spread_confirmed_pullback__mom5_leq0.20_ratio2...,1261,756,29,16,0.022998,0,0.0,0.103133,0.011489,...,True,True,True,0.0,16,3,960,True,True,True
9,trend_pullback_rebound__mom5_leq0.10_mom21_geq...,1261,756,7,4,0.005551,0,0.0,0.103837,0.013093,...,True,True,True,0.0,4,2,960,False,True,False


In [5]:
def full_history_metrics(features: pd.DataFrame, masks: dict[str, pd.Series]) -> pd.DataFrame:
    daily_ret = features["return"].fillna(0.0)
    rows = []
    for name, mask in masks.items():
        mask = mask.reindex(features.index).fillna(False).astype(bool)
        dates = features.index[mask]
        row: dict[str, object] = {
            "candidate": name,
            "full_rows": int(mask.sum()),
            "full_years": int(pd.Index(dates.year).nunique()) if len(dates) else 0,
            "full_start": str(dates.min().date()) if len(dates) else None,
            "full_end": str(dates.max().date()) if len(dates) else None,
            "full_span_days": int((dates.max() - dates.min()).days) if len(dates) > 1 else 0,
        }
        for horizon in (5, 10, 21):
            positive = features.loc[mask, f"forward_return_{horizon}"].dropna()
            negative = features.loc[~mask, f"forward_return_{horizon}"].dropna()
            row[f"full_forward_return_{horizon}_positive_mean"] = float(positive.mean()) if len(positive) else float("nan")
            row[f"full_forward_return_{horizon}_lift"] = (
                float(positive.mean() - negative.mean()) if len(positive) and len(negative) else float("nan")
            )
            active = mask.astype(float).rolling(horizon, min_periods=1).max().shift(1).fillna(0.0)
            uncapped_returns = daily_ret * active
            capped_returns = daily_ret * active * 0.35
            uncapped_nav = (1.0 + uncapped_returns).cumprod()
            capped_nav = (1.0 + capped_returns).cumprod()
            row[f"hold_{horizon}_active_days"] = int(active.sum())
            row[f"hold_{horizon}_total_return"] = float(uncapped_nav.iloc[-1] - 1.0)
            row[f"hold_{horizon}_sharpe"] = annualized_sharpe(uncapped_returns)
            row[f"hold_{horizon}_max_drawdown"] = max_drawdown(uncapped_nav)
            row[f"hold_{horizon}_cap035_total_return"] = float(capped_nav.iloc[-1] - 1.0)
            row[f"hold_{horizon}_cap035_sharpe"] = annualized_sharpe(capped_returns)
            row[f"hold_{horizon}_cap035_max_drawdown"] = max_drawdown(capped_nav)
        row["full_recurrence_gate"] = (
            int(row["full_rows"]) >= 30
            and int(row["full_years"]) >= 5
            and int(row["full_span_days"]) >= 1000
        )
        row["risk_capped_gate"] = float(row["hold_5_cap035_max_drawdown"]) >= -0.20
        rows.append(row)
    return pd.DataFrame(rows)


full_history_signal_metrics = full_history_metrics(full_features, candidate_masks_full)
factor_candidate_scores = target_label_scores.merge(full_history_signal_metrics, on="candidate", how="left")
factor_candidate_scores["direct_current_target_gate"] = (
    factor_candidate_scores["discovery_gate"]
    & factor_candidate_scores["full_recurrence_gate"]
    & (factor_candidate_scores["target_lift_mean"] >= 0.02)
    & (factor_candidate_scores["hold_5_max_drawdown"] >= -0.20)
    & (factor_candidate_scores["full_forward_return_5_lift"] >= 0.03)
)
factor_candidate_scores["target_contract_handoff_gate"] = (
    factor_candidate_scores["discovery_gate"]
    & factor_candidate_scores["full_recurrence_gate"]
    & (factor_candidate_scores["hold_5_max_drawdown"] >= -0.20)
    & (factor_candidate_scores["full_forward_return_5_lift"] >= 0.03)
)
factor_candidate_scores["risk_capped_overlay_gate"] = (
    factor_candidate_scores["discovery_gate"]
    & factor_candidate_scores["full_recurrence_gate"]
    & factor_candidate_scores["risk_capped_gate"]
    & (factor_candidate_scores["full_forward_return_5_lift"] >= 0.03)
)

sort_columns = [
    "direct_current_target_gate",
    "target_contract_handoff_gate",
    "risk_capped_overlay_gate",
    "primary_forward_return_lift",
    "full_forward_return_5_lift",
    "positive_years",
    "full_years",
]
factor_candidate_scores = factor_candidate_scores.sort_values(
    sort_columns,
    ascending=[False, False, False, False, False, False, False],
).reset_index(drop=True)

factor_candidate_scores[[
    "candidate",
    "direct_current_target_gate",
    "target_contract_handoff_gate",
    "risk_capped_overlay_gate",
    "positive_rows",
    "unique_positive_dates",
    "positive_years",
    "primary_forward_return_lift",
    "forward_return_10_lift",
    "target_lift_mean",
    "full_rows",
    "full_years",
    "full_forward_return_5_lift",
    "hold_5_max_drawdown",
    "hold_5_cap035_max_drawdown",
]].head(20)

,candidate,direct_current_target_gate,target_contract_handoff_gate,risk_capped_overlay_gate,positive_rows,unique_positive_dates,positive_years,primary_forward_return_lift,forward_return_10_lift,target_lift_mean,full_rows,full_years,full_forward_return_5_lift,hold_5_max_drawdown,hold_5_cap035_max_drawdown
0,spread_confirmed_pullback__mom5_leq0.20_ratio2...,False,True,True,20,11,3,0.117525,0.051703,0.0,34,13,0.085382,-0.158962,-0.056970
1,trend_pullback_rebound__mom5_leq0.20_mom21_geq...,False,True,True,21,12,3,0.106709,0.063157,0.0,32,12,0.088297,-0.145621,-0.052062
2,spread_confirmed_pullback__mom5_leq0.20_ratio2...,False,False,True,29,16,3,0.091644,0.062854,0.0,55,16,0.059427,-0.406876,-0.155030
3,trend_pullback_rebound__mom5_leq0.20_mom21_geq...,False,False,True,27,15,3,0.079147,0.056064,0.0,42,13,0.063078,-0.330536,-0.126995
4,spread_confirmed_pullback__mom5_leq0.20_ratio2...,False,False,True,41,22,3,0.072626,0.073704,0.0,90,17,0.048760,-0.450002,-0.172392
5,spread_confirmed_pullback__mom5_leq0.15_ratio2...,False,False,True,24,13,3,0.067350,0.109442,0.0,47,16,0.068275,-0.357666,-0.139675
6,trend_pullback_rebound__mom5_leq0.20_mom21_geq...,False,False,True,34,19,3,0.058997,0.051312,0.0,59,15,0.048777,-0.463607,-0.183904
7,trend_pullback_rebound__mom5_leq0.20_mom21_geq...,False,False,True,43,24,3,0.054219,0.049087,0.0,104,17,0.036957,-0.495197,-0.194496
8,trend_pullback_rebound__mom5_leq0.15_mom21_geq...,False,False,True,19,11,3,0.049137,0.083160,0.0,30,9,0.065865,-0.385867,-0.150521
9,trend_pullback_rebound__mom5_leq0.15_mom21_geq...,False,False,True,28,16,3,0.044548,0.069304,0.0,60,16,0.051995,-0.431378,-0.169325


In [6]:
def finite_or_none(value: object) -> object:
    if pd.isna(value):
        return None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    return value


def row_to_jsonable(row: pd.Series) -> dict[str, object]:
    return {str(key): finite_or_none(value) for key, value in row.items()}


selected = factor_candidate_scores.iloc[0]
selected_candidate = str(selected["candidate"])
selected_mask_full = candidate_masks_full[selected_candidate].reindex(full_features.index).fillna(False).astype(bool)
selected_dates = full_features.index[selected_mask_full]
selected_factor_rows = full_features.loc[selected_mask_full, [
    "close",
    "mom_5",
    "mom_21",
    "drawdown_63",
    "ratio_mom_21",
    "soxx_mom_21",
    "rv_21",
    "vix_z_252",
    "forward_return_5",
    "forward_return_10",
    "forward_return_21",
]].reset_index().rename(columns={"date": "datetime"})

if bool(selected["direct_current_target_gate"]):
    decision = "rebound_state_factor_current_target_ready_no_actor_promotion"
    training_allowed = False
    next_action = "build_state_head_training_contract_from_selected_rebound_factor_then_run_walkforward_screen"
elif bool(selected["target_contract_handoff_gate"]):
    decision = "rebound_state_factor_candidate_found_target_contract_needed_no_training"
    training_allowed = False
    next_action = "build_rebound_state_target_contract_from_spread_confirmed_pullback_then_rerun_state_head_label_stability"
elif bool(selected["risk_capped_overlay_gate"]):
    decision = "rebound_state_factor_candidate_found_only_as_risk_capped_overlay_no_training"
    training_allowed = False
    next_action = "test_risk_capped_rebound_overlay_before_state_head_training"
else:
    decision = "rebound_state_factor_screen_no_candidate_no_training"
    training_allowed = False
    next_action = "mine_alternative_rebound_state_source_or_drop_state_head_branch"

summary = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "promotion_allowed": False,
    "training_allowed": training_allowed,
    "source_experiment": SOURCE_FRONTIER,
    "source_target_experiment": "state_objective_fold_native_teacher_router_v1",
    "direct_comparison_baseline": "stable_rebound_label_discovery_v1",
    "panel_source": str(PANEL_PATH.relative_to(ROOT)),
    "target_source": str(TARGET_PATH.relative_to(ROOT)),
    "candidate_count": int(len(factor_candidate_scores)),
    "discovery_gate_candidates": int(factor_candidate_scores["discovery_gate"].sum()),
    "direct_current_target_gate_candidates": int(factor_candidate_scores["direct_current_target_gate"].sum()),
    "target_contract_handoff_gate_candidates": int(factor_candidate_scores["target_contract_handoff_gate"].sum()),
    "risk_capped_overlay_gate_candidates": int(factor_candidate_scores["risk_capped_overlay_gate"].sum()),
    "selected_candidate": selected_candidate,
    "selected_positive_rows": int(selected["positive_rows"]),
    "selected_unique_positive_dates": int(selected["unique_positive_dates"]),
    "selected_positive_years": int(selected["positive_years"]),
    "selected_positive_span_days": int(selected["positive_span_days"]),
    "selected_label_conflict_rate": float(selected["label_conflict_rate"]),
    "selected_target_lift_mean": float(selected["target_lift_mean"]),
    "selected_forward_return_5_lift": float(selected["forward_return_5_lift"]),
    "selected_forward_return_10_lift": float(selected["forward_return_10_lift"]),
    "selected_forward_return_21_lift": float(selected["forward_return_21_lift"]),
    "selected_full_rows": int(selected["full_rows"]),
    "selected_full_years": int(selected["full_years"]),
    "selected_full_span_days": int(selected["full_span_days"]),
    "selected_full_forward_return_5_lift": float(selected["full_forward_return_5_lift"]),
    "selected_hold_5_max_drawdown": float(selected["hold_5_max_drawdown"]),
    "selected_hold_5_cap035_max_drawdown": float(selected["hold_5_cap035_max_drawdown"]),
    "next_mechanism": next_action,
}
selection = {
    **summary,
    "selected_score": row_to_jsonable(selected),
    "selected_full_start": str(selected["full_start"]),
    "selected_full_end": str(selected["full_end"]),
    "target_contract_note": "Current fold-native target lift is still zero for the selected factor, so this is a new target-contract handoff rather than immediate state-head training evidence.",
    "direct_training_note": "No actor is trained or retained by this factor-mining card.",
}

factor_candidate_scores.to_csv(CARD_DIR / "factor_candidate_scores.csv", index=False)
target_label_scores.to_csv(CARD_DIR / "target_label_scores.csv", index=False)
target_fold_metrics.to_csv(CARD_DIR / "target_fold_metrics.csv", index=False)
full_history_signal_metrics.to_csv(CARD_DIR / "full_history_signal_metrics.csv", index=False)
selected_factor_rows.to_csv(CARD_DIR / "selected_factor_rows.csv", index=False)
(CARD_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps({"summary": summary}, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary

{'experiment': 'rebound_state_factor_mining_v1',
 'decision': 'rebound_state_factor_candidate_found_target_contract_needed_no_training',
 'promotion_allowed': False,
 'training_allowed': False,
 'source_experiment': 'state_head_stable_rebound_label_screen_v1',
 'source_target_experiment': 'state_objective_fold_native_teacher_router_v1',
 'direct_comparison_baseline': 'stable_rebound_label_discovery_v1',
 'panel_source': 'data/processed/panels/soxl_soxx_vix_full.csv',
 'target_source': 'runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv',
 'candidate_count': 182,
 'discovery_gate_candidates': 36,
 'direct_current_target_gate_candidates': 0,
 'target_contract_handoff_gate_candidates': 2,
 'risk_capped_overlay_gate_candidates': 12,
 'selected_candidate': 'spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75',
 'selected_positive_rows': 20,
 'selected_unique_positive_dates': 11,
 'selected_positive_years': 3,
 'selected_positive_sp

In [7]:
conclusion = f"""
## Conclusion

Decision: `{summary['decision']}`.

Selected factor: `{summary['selected_candidate']}`.

Target-table evidence: `{summary['selected_positive_rows']}` positive rows, `{summary['selected_unique_positive_dates']}` unique positive dates, `{summary['selected_positive_years']}` positive years, `{summary['selected_positive_span_days']}` span days, duplicate-date conflict rate `{summary['selected_label_conflict_rate']:.4f}`.

Forward-return lift: 5d `{summary['selected_forward_return_5_lift']:.4f}`, 10d `{summary['selected_forward_return_10_lift']:.4f}`, 21d `{summary['selected_forward_return_21_lift']:.4f}`.

Full-history sanity: `{summary['selected_full_rows']}` rows across `{summary['selected_full_years']}` years, 5d lift `{summary['selected_full_forward_return_5_lift']:.4f}`, standalone 5d-hold MaxDD `{summary['selected_hold_5_max_drawdown']:.4f}`, 0.35-capped MaxDD `{summary['selected_hold_5_cap035_max_drawdown']:.4f}`.

Training decision: no actor training in this card. The current fold-native target lift is `{summary['selected_target_lift_mean']:.4f}`, so the selected factor should become a new rebound-state target contract and then rerun label stability before state-head training.

Next action: `{summary['next_mechanism']}`.

Artifacts: `factor_candidate_scores.csv`, `target_label_scores.csv`, `target_fold_metrics.csv`, `full_history_signal_metrics.csv`, `selected_factor_rows.csv`, `selection.json`, `summary.json`.
""".strip()
display(Markdown(conclusion))

## Conclusion

Decision: `rebound_state_factor_candidate_found_target_contract_needed_no_training`.

Selected factor: `spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75`.

Target-table evidence: `20` positive rows, `11` unique positive dates, `3` positive years, `842` span days, duplicate-date conflict rate `0.0000`.

Forward-return lift: 5d `0.1175`, 10d `0.0517`, 21d `0.2338`.

Full-history sanity: `34` rows across `13` years, 5d lift `0.0854`, standalone 5d-hold MaxDD `-0.1590`, 0.35-capped MaxDD `-0.0570`.

Training decision: no actor training in this card. The current fold-native target lift is `0.0000`, so the selected factor should become a new rebound-state target contract and then rerun label stability before state-head training.

Next action: `build_rebound_state_target_contract_from_spread_confirmed_pullback_then_rerun_state_head_label_stability`.

Artifacts: `factor_candidate_scores.csv`, `target_label_scores.csv`, `target_fold_metrics.csv`, `full_history_signal_metrics.csv`, `selected_factor_rows.csv`, `selection.json`, `summary.json`.